# Notebook 18 — Instruction fine-tuning passo passo

**Modulo:** 18 — Instruction fine-tuning passo passo
**Tempo stimato:** 2-3 ore (esecuzione su CPU, GPU consigliata ma non obbligatoria)

## Cosa costruirai

Al termine di questo notebook avrai:
- Un dataset italiano di 38 coppie testo-riassunto, scritto a mano, diviso in
  training e validation
- La pipeline completa di preprocessing: template prompt-completion,
  tokenizzazione, padding dinamico, sostituzione a -100 (Modulo 18, § 5-6)
- Un vero **full fine-tuning** di Flan-T5-small con `Seq2SeqTrainer` di
  Hugging Face, con iperparametri di addestramento configurati
  esplicitamente (Modulo 18, § 7)
- Un confronto concreto **prima vs dopo** il fine-tuning sugli stessi due
  esempi, mai visti né in training né in validation
- Una funzione di pipeline riutilizzabile che estendi tu stesso nella sfida
  finale

## Prerequisiti

```bash
pip install transformers datasets sentencepiece accelerate torch python-dotenv
```

Consigliato eseguire su Colab (runtime CPU va bene; con GPU è più veloce, ma
il dataset è piccolo e l'addestramento richiede comunque pochi minuti anche
su CPU). Non serve nessuna chiave API: lavoriamo solo con modelli pubblici
scaricati da Hugging Face.

Se vuoi evitare limiti di download anonimi di Hugging Face, puoi mettere un
token opzionale nel tuo `.env`:
```
HUGGINGFACE_TOKEN=hf_...
```
Non è indispensabile per questo notebook.

---
## Cella 0 — Setup

Verifichiamo le librerie e un eventuale token Hugging Face opzionale.

In [ ]:
import os
import random
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")  # opzionale per questo notebook

def verifica_chiave(nome: str, valore: str | None) -> None:
    if valore:
        print(f"OK {nome}: {valore[:8]}...")
    else:
        print(f"-- {nome}: non impostata (va bene, non serve per i modelli pubblici di questo notebook)")

verifica_chiave("HUGGINGFACE_TOKEN", HF_TOKEN)

import transformers
import datasets
print(f"\ntransformers versione: {transformers.__version__}")
print(f"datasets versione:     {datasets.__version__}")

random.seed(42)

---
## Cella 1 — Il compito: riassunto istruzionale in italiano

Seguiamo esattamente lo schema del Modulo 18, § 5: definiamo un **template
prompt → completion** fisso, applicato in modo identico a tutti gli esempi
del dataset. Il compito è il riassunto (*summarization*) di brevi paragrafi
informativi in italiano, in una singola frase.

```
Prompt:     "Riassumi il seguente testo:\n\n{testo}"
Completion: {riassunto}
```

Definiamo anche due esempi di controllo qualitativo, **volutamente esclusi**
dal dataset di training e di validation: li useremo solo per il confronto
prima/dopo, per essere sicuri di misurare la capacità di generalizzazione
del modello e non la memorizzazione.

In [ ]:
def costruisci_prompt(testo: str) -> str:
    """Applica il template prompt-completion del Modulo 18, § 5."""
    return f"Riassumi il seguente testo:\n\n{testo}"


# Esempi di controllo qualitativo: MAI inclusi nel dataset di training/validation.
# Argomenti scelti apposta diversi da quelli del dataset (droni civili, Machu Picchu).
ESEMPI_QUALITATIVI = [
    {
        "testo": (
            "I droni per uso civile vengono impiegati sempre più spesso per il "
            "monitoraggio agricolo, la consegna di pacchi e le riprese fotografiche. "
            "Le normative sul loro utilizzo variano molto da paese a paese."
        ),
        "riassunto_atteso": "I droni civili si usano per agricoltura, consegne e fotografia, con normative diverse tra paesi.",
    },
    {
        "testo": (
            "Il Machu Picchu, antica città Inca situata sulle Ande peruviane, fu "
            "riscoperta dal mondo occidentale solo nel 1911. Oggi è patrimonio "
            "UNESCO e una delle mete turistiche più ambite del Sud America."
        ),
        "riassunto_atteso": "Il Machu Picchu, città Inca riscoperta nel 1911, è oggi patrimonio UNESCO molto visitato.",
    },
]

for e in ESEMPI_QUALITATIVI:
    print(costruisci_prompt(e["testo"])[:60], "...")

---
## Cella 2 — Comportamento di Flan-T5-small PRIMA del fine-tuning su questo task

**Attenzione a non confondere due cose diverse:** Flan-T5-small (Modulo 17)
è **già** stato sottoposto a instruction fine-tuning su oltre mille compiti
generici, quindi sa già "cosa vuol dire" seguire un'istruzione. Quello che
**non ha mai visto** è il nostro stile specifico di riassunto in una sola
frase su questo dominio di testi. Il fine-tuning che facciamo in questo
notebook è una **seconda specializzazione**, più mirata, sopra quella già
esistente — esattamente il caso descritto nel Modulo 18, § 1.

**Prima di eseguire:** ti aspetti un riassunto già buono, o generico/troppo
lungo rispetto a quanto vorremmo?

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NOME_MODELLO = "google/flan-t5-small"

tokenizer_base = AutoTokenizer.from_pretrained(NOME_MODELLO)
model_base = AutoModelForSeq2SeqLM.from_pretrained(NOME_MODELLO)

def genera_riassunto(tokenizer, model, testo: str, max_new_tokens: int = 40) -> str:
    """Applica il template, tokenizza, genera, decodifica."""
    prompt = costruisci_prompt(testo)
    input_ids = tokenizer(prompt, return_tensors="pt", truncation=True).input_ids
    output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("=== Flan-T5-small, PRIMA del fine-tuning specifico ===\n")
for e in ESEMPI_QUALITATIVI:
    generato = genera_riassunto(tokenizer_base, model_base, e["testo"])
    print(f"Riassunto atteso:  {e['riassunto_atteso']}")
    print(f"Riassunto generato: {generato}\n")

**Cosa osservare:** con tutta probabilità il modello genera qualcosa di
plausibile ma spesso troppo lungo, troppo simile a una copia parziale del
testo originale, o con uno stile diverso da quello conciso dei nostri
riassunti di riferimento. Non è "sbagliato" in senso assoluto: è solo
**non adattato al nostro formato specifico**. È esattamente il problema che
il fine-tuning di questo notebook risolve.

---
## Cella 3 — Il dataset: 38 coppie testo-riassunto

Seguendo il Modulo 18, § 4: per un singolo compito, poche centinaia di
esempi sono già indicativi in un progetto reale. Qui usiamo un dataset
volutamente **minuscolo** (38 esempi) per restare eseguibile in pochi
minuti su CPU: è sufficiente per vedere un cambiamento di comportamento
misurabile, non per ottenere un modello di produzione.

In [ ]:
DATASET = [
    ("Il telescopio spaziale Hubble orbita attorno alla Terra dal 1990 e ha permesso di osservare galassie lontanissime con un dettaglio mai raggiunto prima. Le sue immagini hanno cambiato profondamente la comprensione dell'universo.",
     "Hubble osserva l'universo dal 1990 con immagini di dettaglio senza precedenti."),
    ('La Rivoluzione Industriale, iniziata nel XVIII secolo in Inghilterra, trasformò le economie agricole in economie basate sulla fabbrica e sulla produzione di massa. Portò enormi cambiamenti sociali, ma anche nuove forme di sfruttamento del lavoro.',
     "La Rivoluzione Industriale trasformò l'economia agricola in economia industriale, con grandi costi sociali."),
    ('Il Colosseo, costruito a Roma nel I secolo d.C., poteva ospitare fino a 50.000 spettatori durante i combattimenti dei gladiatori. Oggi è uno dei monumenti più visitati al mondo.',
     'Il Colosseo, anfiteatro romano del I secolo, è oggi un monumento tra i più visitati al mondo.'),
    ("Le api svolgono un ruolo fondamentale nell'impollinazione delle piante da fiore e delle colture agricole. Il loro declino, causato da pesticidi e perdita di habitat, preoccupa scienziati e agricoltori in tutto il mondo.",
     "Le api sono essenziali per l'impollinazione e il loro declino preoccupa scienziati e agricoltori."),
    ('Il Monte Everest, con i suoi 8.849 metri, è la montagna più alta del mondo e si trova al confine tra Nepal e Tibet. Ogni anno centinaia di alpinisti tentano la scalata, nonostante i rischi elevati.',
     "L'Everest, la montagna più alta del mondo, attira ogni anno centinaia di alpinisti nonostante i rischi."),
    ("L'intelligenza artificiale generativa ha reso possibile creare testi, immagini e musica a partire da semplici istruzioni testuali. Questa tecnologia sta trasformando settori come l'editoria, il design e l'intrattenimento.",
     "L'IA generativa crea contenuti da istruzioni testuali e sta trasformando editoria, design e intrattenimento."),
    ("La dieta mediterranea, basata su olio d'oliva, verdura, pesce e cereali integrali, è associata a un minor rischio di malattie cardiovascolari. Numerosi studi scientifici ne confermano i benefici per la salute.",
     'La dieta mediterranea è associata a un minor rischio di malattie cardiovascolari.'),
    ("Il fiume Nilo, lungo oltre 6.600 chilometri, attraversa undici paesi africani ed è da millenni fonte di vita per le popolazioni che abitano le sue rive. L'antico Egitto si sviluppò proprio grazie alle sue acque.",
     "Il Nilo attraversa undici paesi africani ed è stato fonte di vita fin dall'antico Egitto."),
    ('La Grande Barriera Corallina australiana è il più esteso sistema di barriere coralline al mondo, visibile persino dallo spazio. Il riscaldamento degli oceani sta però causando un preoccupante sbiancamento dei coralli.',
     'La Grande Barriera Corallina, visibile dallo spazio, è minacciata dallo sbiancamento causato dal riscaldamento degli oceani.'),
    ("Leonardo da Vinci fu pittore, inventore e scienziato nell'Italia del Rinascimento. Oltre a capolavori come la Gioconda, lasciò migliaia di disegni tecnici che anticipavano macchine costruite secoli dopo.",
     'Leonardo da Vinci fu artista e inventore del Rinascimento, con progetti che anticiparono il futuro.'),
    ('Il mercato delle auto elettriche è cresciuto rapidamente negli ultimi anni, sostenuto da incentivi statali e dalla riduzione dei costi delle batterie. Molti produttori hanno annunciato la fine della produzione di motori a combustione entro il prossimo decennio.',
     'Il mercato delle auto elettriche cresce grazie a incentivi statali e batterie più economiche.'),
    ("Lo yoga è una pratica millenaria originaria dell'India che combina esercizi fisici, controllo del respiro e meditazione. Oggi è diffuso in tutto il mondo come metodo per ridurre lo stress e migliorare il benessere generale.",
     'Lo yoga, pratica millenaria indiana, è oggi diffuso nel mondo per ridurre lo stress.'),
    ('La squadra italiana di pallavolo ha vinto il campionato europeo dopo una finale combattuta terminata al quinto set. Migliaia di tifosi hanno festeggiato nelle piazze delle principali città italiane.',
     "L'Italia ha vinto il campionato europeo di pallavolo dopo una finale al quinto set."),
    ("Gli oceani coprono oltre il 70% della superficie terrestre e producono più della metà dell'ossigeno presente nell'atmosfera, soprattutto grazie al fitoplancton. La loro salute è quindi cruciale per l'intero pianeta.",
     "Gli oceani coprono gran parte della Terra e producono oltre metà dell'ossigeno atmosferico."),
    ('La stampa a caratteri mobili, inventata da Gutenberg nel XV secolo, rivoluzionò la diffusione della conoscenza in Europa. Per la prima volta i libri poterono essere prodotti in grandi quantità a costi contenuti.',
     "L'invenzione della stampa di Gutenberg rese possibile la diffusione dei libri su larga scala."),
    ("I pinguini imperatore vivono in Antartide e resistono a temperature che possono scendere sotto i -60 gradi. I maschi si stringono in grandi gruppi per proteggere l'unico uovo durante l'inverno polare.",
     "I pinguini imperatore sopravvivono alle temperature estreme dell'Antartide proteggendo l'uovo in gruppo."),
    ('Il turismo spaziale privato ha compiuto passi importanti negli ultimi anni, con diverse aziende che offrono voli suborbitali a pagamento. I costi restano però proibitivi per la maggior parte delle persone.',
     'Il turismo spaziale privato è in crescita ma i costi restano molto elevati.'),
    ("La Silicon Valley, in California, ospita le sedi di alcune delle aziende tecnologiche più importanti al mondo. Il suo successo si deve alla combinazione di università d'eccellenza, capitale di rischio e cultura dell'innovazione.",
     "La Silicon Valley deve il suo successo a università, capitali di rischio e cultura dell'innovazione."),
    ("Il cioccolato viene prodotto a partire dai semi di cacao, originari dell'America centrale e utilizzati fin dall'epoca precolombiana. Oggi la maggior parte del cacao mondiale viene coltivata in Africa occidentale.",
     'Il cioccolato deriva dal cacao, oggi coltivato soprattutto in Africa occidentale.'),
    ("Il riscaldamento globale sta causando lo scioglimento dei ghiacciai in tutto il mondo, con un impatto diretto sull'innalzamento del livello del mare. Gli scienziati chiedono azioni immediate per ridurre le emissioni di gas serra.",
     'Il riscaldamento globale scioglie i ghiacciai e fa salire il livello del mare.'),
    ('La Grande Muraglia Cinese si estende per migliaia di chilometri e fu costruita in diverse epoche per difendere il territorio cinese dalle invasioni. È oggi uno dei monumenti più famosi al mondo.',
     'La Grande Muraglia Cinese, costruita per scopi difensivi, è oggi un monumento famoso in tutto il mondo.'),
    ('I vaccini a mRNA, sviluppati e diffusi su larga scala a partire dal 2020, hanno rappresentato una svolta nella lotta contro le malattie infettive. La tecnologia è ora studiata anche per altre applicazioni mediche.',
     'I vaccini a mRNA hanno segnato una svolta e sono ora studiati per nuove applicazioni mediche.'),
    ("Il Giappone è noto per l'elevata puntualità dei suoi treni ad alta velocità, gli Shinkansen, in circolazione dal 1964. Il ritardo medio annuo è di pochi secondi su migliaia di corse.",
     'Gli Shinkansen giapponesi, in servizio dal 1964, sono famosi per la puntualità estrema.'),
    ("La biblioteca di Alessandria, fondata nell'antico Egitto, fu per secoli uno dei principali centri di sapere del mondo antico. La sua distruzione è considerata una delle maggiori perdite culturali della storia.",
     'La biblioteca di Alessandria fu un grande centro di sapere antico, la cui perdita fu enorme.'),
    ('Il caffè è una delle bevande più consumate al mondo e la sua coltivazione dà lavoro a milioni di persone, soprattutto in Sud America, Africa e Asia. Il commercio equo cerca di garantire condizioni più giuste ai coltivatori.',
     'Il caffè dà lavoro a milioni di persone e il commercio equo mira a condizioni più giuste.'),
    ('I robot aspirapolvere utilizzano sensori e algoritmi di mappatura per pulire gli ambienti domestici in modo autonomo. I modelli più recenti riescono a evitare ostacoli e a tornare da soli alla base di ricarica.',
     'I robot aspirapolvere puliscono in autonomia grazie a sensori e algoritmi di mappatura.'),
    ("L'Amazzonia, la più grande foresta pluviale del mondo, produce una quantità enorme di ossigeno e ospita una biodiversità straordinaria. La deforestazione minaccia però il suo equilibrio ecologico.",
     "L'Amazzonia produce molto ossigeno e ospita grande biodiversità, ma è minacciata dalla deforestazione."),
    ('Le biblioteche pubbliche moderne offrono non solo libri, ma anche accesso a internet, corsi di formazione e spazi di coworking. Sono diventate centri culturali polifunzionali per le comunità locali.',
     'Le biblioteche pubbliche moderne offrono servizi digitali e formativi oltre ai libri.'),
    ("Il Partenone, costruito ad Atene nel V secolo a.C., è uno dei simboli più celebri dell'antica Grecia. Dedicato alla dea Atena, rappresenta l'apice dell'architettura classica.",
     "Il Partenone di Atene, del V secolo a.C., è un simbolo dell'architettura classica greca."),
    ('Le energie rinnovabili, come solare ed eolico, stanno diventando sempre più competitive rispetto ai combustibili fossili grazie alla riduzione dei costi tecnologici. Molti paesi puntano a raggiungere la neutralità carbonica entro metà secolo.',
     'Le energie rinnovabili diventano più competitive e molti paesi puntano alla neutralità carbonica.'),
    ('La maratona di New York è una delle corse podistiche più famose al mondo e attira ogni anno decine di migliaia di partecipanti da tutti i continenti. Il percorso attraversa i cinque distretti della città.',
     'La maratona di New York attira decine di migliaia di partecipanti da tutto il mondo.'),
    ('Il DNA contiene le istruzioni genetiche necessarie per lo sviluppo e il funzionamento di ogni essere vivente. La scoperta della sua struttura a doppia elica, nel 1953, è considerata una pietra miliare della biologia.',
     'La scoperta della struttura del DNA nel 1953 fu una pietra miliare della biologia.'),
    ('I social media hanno cambiato profondamente il modo in cui le persone comunicano e si informano, ma hanno anche sollevato preoccupazioni legate alla disinformazione e alla salute mentale, soprattutto tra i più giovani.',
     'I social media hanno cambiato la comunicazione ma sollevano preoccupazioni su disinformazione e salute mentale.'),
    ("Il vino viene prodotto attraverso la fermentazione del succo d'uva ed è parte della cultura di molte regioni del mondo, dall'Italia alla Francia fino al Cile. Ogni territorio sviluppa varietà e tecniche proprie.",
     "Il vino nasce dalla fermentazione dell'uva ed è parte della cultura di molte regioni del mondo."),
    ('Le missioni spaziali verso Marte cercano tracce di vita passata e studiano la possibilità di future missioni con equipaggio umano. I rover hanno già analizzato rocce e atmosfera del pianeta rosso.',
     'Le missioni su Marte cercano tracce di vita passata e preparano future missioni umane.'),
    ("Il flamenco è una forma d'arte spagnola che unisce canto, chitarra e danza, originaria dell'Andalusia. È stato riconosciuto dall'UNESCO come patrimonio culturale immateriale dell'umanità.",
     'Il flamenco, arte andalusa di canto e danza, è patrimonio culturale UNESCO.'),
    ("Le città smart utilizzano sensori e dati in tempo reale per ottimizzare traffico, illuminazione pubblica e gestione dei rifiuti. L'obiettivo è rendere i servizi urbani più efficienti e sostenibili.",
     'Le città smart usano sensori e dati per rendere i servizi urbani più efficienti.'),
    ("La Torre Eiffel, costruita a Parigi nel 1889 per l'Esposizione Universale, doveva essere smontata dopo pochi anni ma divenne il simbolo della città. Oggi è una delle attrazioni turistiche più visitate al mondo.",
     'La Torre Eiffel, costruita nel 1889, è oggi uno dei simboli più visitati al mondo.'),
]

print(f"Numero totale di esempi nel dataset: {len(DATASET)}")

---
## Cella 4 — Split train/validation

Modulo 18, § 3, passo 2: dividiamo il dataset per poter misurare la
generalizzazione, non solo la memorizzazione. Con un dataset così piccolo
non ritagliamo anche un test set separato: usiamo direttamente
`ESEMPI_QUALITATIVI` (Cella 1) come controllo finale, tenuto fuori fin
dall'inizio.

In [ ]:
from datasets import Dataset

indici = list(range(len(DATASET)))
random.shuffle(indici)

N_VALIDATION = 8
indici_validation = set(indici[:N_VALIDATION])

esempi_train = [{"testo": t, "riassunto": r} for i, (t, r) in enumerate(DATASET) if i not in indici_validation]
esempi_validation = [{"testo": t, "riassunto": r} for i, (t, r) in enumerate(DATASET) if i in indici_validation]

ds = {
    "train": Dataset.from_list(esempi_train),
    "validation": Dataset.from_list(esempi_validation),
}

print(f"Esempi di training:   {len(ds['train'])}")
print(f"Esempi di validation: {len(ds['validation'])}")

---
## Cella 5 — Costruzione dei prompt e tokenizzazione

Modulo 18, § 5-6: applichiamo il template a ogni esempio, poi tokenizziamo
prompt e completion **separatamente**. Il prompt diventa `input_ids`, il
completion diventa `labels`.

**TODO:** completa la tokenizzazione del completion (riassunto): manca la
riga che lo trasforma in `labels`.

In [ ]:
MAX_LEN_PROMPT = 256      # ampio margine: i nostri testi sono corti
MAX_LEN_COMPLETION = 48   # i riassunti sono una sola frase

def preprocessa(esempio):
    prompt = costruisci_prompt(esempio["testo"])

    model_inputs = tokenizer_base(
        prompt, max_length=MAX_LEN_PROMPT, truncation=True
    )

    # TODO: tokenizza esempio["riassunto"] con tokenizer_base, usando
    # max_length=MAX_LEN_COMPLETION e truncation=True, e assegna il
    # risultato a model_inputs["labels"] (come lista di input_ids, non
    # come dict completo)
    labels = None  # <-- sostituisci con la tokenizzazione del completamento

    model_inputs["labels"] = labels
    return model_inputs


ds_tok = {
    "train": ds["train"].map(preprocessa, remove_columns=["testo", "riassunto"]),
    "validation": ds["validation"].map(preprocessa, remove_columns=["testo", "riassunto"]),
}

print(ds_tok["train"][0])

---
## Cella 6 — Data collator: padding dinamico e sostituzione a -100

Modulo 18, § 6: le sequenze di un batch devono avere la stessa lunghezza.
`DataCollatorForSeq2Seq` fa **padding dinamico** (in base al batch corrente,
non a un massimo fisso) e sostituisce automaticamente il padding delle
`labels` con **-100**, per farlo ignorare dalla funzione di errore.

In [ ]:
from transformers import DataCollatorForSeq2Seq

# Carichiamo un modello "fresco" dedicato al training: model_base resta
# intatto, così potremo confrontare prima/dopo senza doverlo ricaricare.
model = AutoModelForSeq2SeqLM.from_pretrained(NOME_MODELLO)

data_collator = DataCollatorForSeq2Seq(
    tokenizer_base,
    model=model,
    label_pad_token_id=-100,
)

print("Data collator pronto. Il modello di training è un'istanza separata da model_base.")

---
## Cella 7 — Iperparametri di addestramento

Modulo 18, § 7. Ricapitolando la tabella della dispensa: qui configuriamo
**iperparametri di addestramento** (`learning_rate`, `batch_size`,
`num_train_epochs`), non gli iperparametri di inferenza del Modulo 09.

Una nota pratica: la dispensa indica 3-4 epoch come tipico per il
fine-tuning. Qui usiamo **8 epoch** perché il nostro dataset di training è
volutamente minuscolo (30 esempi): con così pochi esempi, poche epoch non
bastano a produrre un cambiamento osservabile. In un progetto reale, con
centinaia/migliaia di esempi, torneresti verso 3-4 epoch, per il motivo
spiegato nel Modulo 18, § 4 (rischio di dimenticanza catastrofica /
overfitting).

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

DIRECTORY_OUTPUT = "./flan-t5-riassunti-it"

training_args = Seq2SeqTrainingArguments(
    output_dir=DIRECTORY_OUTPUT,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=MAX_LEN_COMPLETION,
    learning_rate=3e-4,
    num_train_epochs=8,
    eval_strategy="epoch",        # in transformers < 4.41 si chiamava evaluation_strategy
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_strategy="epoch",
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer_base,
)

print("Trainer configurato. Esempi di training per epoch:", len(ds_tok["train"]))

---
## Cella 8 — Avviamo l'addestramento

Con un dataset così piccolo, l'addestramento richiede pochi minuti anche su
CPU (Modulo 18, § 8: la maggior parte dello sforzo reale è a monte, non qui).
`load_best_model_at_end=True` fa sì che, al termine, `model` contenga i pesi
del checkpoint con la validation loss migliore, non necessariamente
l'ultimo.

In [ ]:
trainer.train()

print("\n=== Andamento della loss per epoch ===")
for log in trainer.state.log_history:
    if "eval_loss" in log:
        print(f"epoch {log['epoch']:.0f}: eval_loss = {log['eval_loss']:.4f}")
    elif "loss" in log:
        print(f"epoch {log['epoch']:.0f}: training loss = {log['loss']:.4f}")

---
## Cella 9 — Esperimento guidato: confronto PRIMA vs DOPO

Riusiamo esattamente gli stessi `ESEMPI_QUALITATIVI` della Cella 1-2, mai
inclusi nel training né nel validation. `model_base` non è mai stato
toccato dal training; `model` è ora l'istanza fine-tuned.

In [ ]:
print(f"{'ATTESO':60} | {'PRIMA (model_base)':55} | {'DOPO (model fine-tuned)'}")
print("-" * 170)

for e in ESEMPI_QUALITATIVI:
    out_prima = genera_riassunto(tokenizer_base, model_base, e["testo"])
    out_dopo = genera_riassunto(tokenizer_base, model, e["testo"])
    print(f"{e['riassunto_atteso'][:58]:60} | {out_prima[:53]:55} | {out_dopo}")

**Le tue osservazioni** (modifica questa cella):

- Il riassunto "prima" era: ...
- Il riassunto "dopo" è: ...
- È più vicino allo stile atteso (una frase, concisa)? ...
- Quanto ti fidi di questo giudizio "a occhio"? Nel Modulo 19 vedremo come
  quantificarlo con ROUGE e BLEU invece di valutarlo solo a occhio.

---
## Esercizio — completa il codice

**TODO:** scrivi un tuo testo breve (2-3 frasi, un argomento **non**
presente né nel dataset né in `ESEMPI_QUALITATIVI`) e confronta l'output di
`model_base` e `model` con `genera_riassunto`.

In [ ]:
# TODO: scrivi qui il tuo testo
mio_testo = ""  # es. un paragrafo su un argomento a tua scelta

# TODO: chiama genera_riassunto() con model_base e con model, e stampa entrambi i risultati


---
## Sfida finale — estendi il dataset e ripeti la pipeline end-to-end

Costruisci una funzione `pipeline_completa` che integra **tutti** i passi
visti in questo notebook: split, tokenizzazione, training, valutazione
qualitativa. Poi usala per verificare se aggiungere dati migliora il
comportamento del modello.

1. Aggiungi almeno **5 nuovi esempi** (testo, riassunto) a
   `NUOVI_ESEMPI`, su un dominio a tua scelta (es. cucina, musica, sport
   diversi da quelli già presenti).
2. Completa i punti marcati con `# TODO` nella funzione `pipeline_completa`.
3. Esegui la pipeline sul dataset esteso (`DATASET + NUOVI_ESEMPI`) e
   confronta i risultati su `ESEMPI_QUALITATIVI` con quelli ottenuti prima.

In [ ]:
NUOVI_ESEMPI = [
    # TODO: aggiungi almeno 5 tuple (testo, riassunto) come quelle di DATASET
]


def pipeline_completa(coppie: list[tuple[str, str]], num_epochs: int = 8, n_validation: int = 8):
    """Esegue l'intera pipeline di fine-tuning su un nuovo insieme di coppie
    (testo, riassunto) e restituisce il modello addestrato.
    """
    # TODO 1: crea indici, mescolali con random.shuffle, e dividi in
    # esempi_train / esempi_validation come nella Cella 4 (usa n_validation)

    # TODO 2: costruisci ds_train e ds_validation con Dataset.from_list

    # TODO 3: applica preprocessa() con .map(..., remove_columns=["testo", "riassunto"])
    # a entrambi i sottoinsiemi (riusa la funzione preprocessa già definita)

    # TODO 4: carica un nuovo modello con AutoModelForSeq2SeqLM.from_pretrained(NOME_MODELLO)
    # e crea un nuovo DataCollatorForSeq2Seq per quel modello

    # TODO 5: crea Seq2SeqTrainingArguments (puoi riusare gli stessi valori della Cella 7,
    # con num_train_epochs=num_epochs) e un nuovo Seq2SeqTrainer

    # TODO 6: chiama trainer.train() e restituisci il modello addestrato
    pass


# Scommenta quando hai completato i TODO sopra:
# dataset_esteso = DATASET + NUOVI_ESEMPI
# nuovo_modello = pipeline_completa(dataset_esteso, num_epochs=8)
#
# print("\n=== Confronto sui soliti ESEMPI_QUALITATIVI, con il modello esteso ===")
# for e in ESEMPI_QUALITATIVI:
#     print(genera_riassunto(tokenizer_base, nuovo_modello, e["testo"]))

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 18 — Modulo 18 del Corso Completo di IA Generativa*